In [ ]:
import os
import json
import itertools
import random
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, CSVLogger

SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [14]:
# ── Konfiguration ──────────────────────────────────────────────────────────────
# Passe diesen Pfad bei Bedarf an.
DATA_DIR = Path("../../data/Apple_Data_bereinigt")

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
SEED       = 42



In [ ]:
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"DATA_DIR existiert nicht: {DATA_DIR}\n"
    )

train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.3,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

temp_ds_raw = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.3,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds_raw.class_names
num_classes = len(class_names)

# Validation-Teil wird halbiert: 15% Validation, 15% Test
# Wichtig: Testdaten werden erst ganz am Ende zur Bewertung verwendet.
temp_batches = temp_ds_raw.cardinality().numpy()
val_batches = temp_batches // 2

val_ds_raw = temp_ds_raw.take(val_batches)
test_ds_raw = temp_ds_raw.skip(val_batches)

print("Klassen:", class_names)
print("Anzahl Klassen:", num_classes)
print("Training Batches:", train_ds_raw.cardinality().numpy())
print("Validation Batches:", val_ds_raw.cardinality().numpy())
print("Test Batches:", test_ds_raw.cardinality().numpy())
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.1),
], name="augmentation")

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds_raw.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

val_ds = val_ds_raw.prefetch(AUTOTUNE)
test_ds = test_ds_raw.prefetch(AUTOTUNE)

Found 4751 files belonging to 2 classes.
Using 3326 files for training.
Found 4751 files belonging to 2 classes.
Using 1425 files for validation.
Klassen: ['Apple__Healthy', 'Apple__Rotten']
Anzahl Klassen: 2
Training Batches: 104
Validation Batches: 22
Test Batches: 23


In [24]:
from tensorflow.keras.preprocessing import image as keras_image
import xgboost as xgb
import joblib

#LADEN eigenes mini cnn
model_simpel = tf.keras.models.load_model("../../modelle/final/eigenesCNNmod.keras")

#LADEN randomforest und xg boost (+laden mobilenetv2, +extrahierte features)
#mobilenet_extr = tf.keras.models.load_model("../../modelle/final/mobilenet_extractor.keras")
xgb_model = xgb.XGBClassifier()
xgb_model.load_model("../../modelle/final/mobilenet_xgboost.json") 
model_randomforest = joblib.load("../../modelle/final/rf_classifier.joblib")

X_train = np.load("../../modelle/feature/X_train.npy")
y_train = np.load("../../modelle/feature/y_train.npy")
X_val = np.load("../../modelle/feature/X_val.npy")
y_val = np.load("../../modelle/feature/y_val.npy")

#LADEN optimiertes pretrained CNN
model_best_CNN =  tf.keras.models.load_model("../../modelle/final/MobileNetV2ohneFT.keras")

#LADEN optimiertes finetuned pretrained CNN
model_finetuned = tf.keras.models.load_model("../../modelle/final/MobileNetV2FT.keras")

In [25]:
import time
MODEL_PATH = Path("../../modelle/final/eigenesCNNmod.keras")
model_simpel = tf.keras.models.load_model(MODEL_PATH)
model = model_simpel
print("👾model_simpel👾")
print("=== Time ===")
images_batch, _ = next(iter(test_ds))
# Warm-up: Feature-Extraktion + Random-Forest-Vorhersage
for _ in range(5):
    model.predict(images_batch, verbose=0)


n_runs = 100
start = time.time()

for _ in range(n_runs):
    model.predict(images_batch, verbose=0)

end = time.time()

# Zeit pro Bild, nicht nur pro Batch:
avg_time_per_image = (end - start) / (n_runs * len(images_batch))

print(f"Durchschnittliche Laufzeit pro Bild: {avg_time_per_image * 1000:.2f} ms")
y_true, y_pred, y_conf = [], [], []

def get_file_size_mb(path):
    return os.path.getsize(path) / (1024**2)
print("=== Size ===")
size_mb = get_file_size_mb(MODEL_PATH)
print(f"{size_mb:.2f} MB")

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))
    y_conf.extend(np.max(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_conf = np.array(y_conf)

print("=== Classification Report ===")
print(classification_report(y_true, y_pred, target_names=class_names,
    digits=4))
cm = confusion_matrix(y_true, y_pred)

print(cm)

👾model_simpel👾
=== Time ===
Durchschnittliche Laufzeit pro Bild: 7.55 ms
=== Size ===
0.89 MB
=== Classification Report ===
                precision    recall  f1-score   support

Apple__Healthy     0.9368    0.8316    0.8810       285
 Apple__Rotten     0.8974    0.9633    0.9292       436

      accuracy                         0.9112       721
     macro avg     0.9171    0.8974    0.9051       721
  weighted avg     0.9130    0.9112    0.9102       721

[[237  48]
 [ 16 420]]


In [36]:
def modellgroesse_ohne_optimizer(model):
    weights = model.get_weights()
    parameter = sum(w.size for w in weights)
    speicher_mib = sum(w.nbytes for w in weights) / (1024 ** 2)
    return parameter, speicher_mib

In [37]:
import time
model = model_best_CNN
print("👾Pretrained CNN👾")
print("=== Time ===")
images_batch, _ = next(iter(test_ds))
# Warm-up: Feature-Extraktion + Random-Forest-Vorhersage
for _ in range(5):
    model.predict(images_batch, verbose=0)


n_runs = 100
start = time.time()

for _ in range(n_runs):
    model.predict(images_batch, verbose=0)

end = time.time()

# Zeit pro Bild, nicht nur pro Batch:
avg_time_per_image = (end - start) / (n_runs * len(images_batch))

print(f"Durchschnittliche Laufzeit pro Bild: {avg_time_per_image * 1000:.2f} ms")
y_true, y_pred, y_conf = [], [], []

def get_file_size_mb(path):
    return os.path.getsize(path) / (1024**2)
print("=== Size ===")
size_mb = get_file_size_mb("../../modelle/final/MobileNetV2ohneFT.keras")
print(f"{size_mb:.2f} MB")

print("=== Size ohne Optimierer ===")
parameter, size_mib = modellgroesse_ohne_optimizer(model_best_CNN)

print(f"{size_mib:.2f} MB")

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))
    y_conf.extend(np.max(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_conf = np.array(y_conf)

print("=== Classification Report ===")
print(classification_report(y_true, y_pred, target_names=class_names,
    digits=4))
cm = confusion_matrix(y_true, y_pred)


print(cm)

👾Pretrained CNN👾
=== Time ===
Durchschnittliche Laufzeit pro Bild: 29.86 ms
=== Size ===
12.99 MB
=== Size ohne Optimierer ===
9.89 MB
=== Classification Report ===
                precision    recall  f1-score   support

Apple__Healthy     0.9862    0.9862    0.9862       290
 Apple__Rotten     0.9907    0.9907    0.9907       431

      accuracy                         0.9889       721
     macro avg     0.9885    0.9885    0.9885       721
  weighted avg     0.9889    0.9889    0.9889       721

[[286   4]
 [  4 427]]


In [38]:
import time
model = model_finetuned
print("👾Pretained finetuned CNN👾")
print("=== Time ===")
images_batch, _ = next(iter(test_ds))
# Warm-up: Feature-Extraktion + Random-Forest-Vorhersage
for _ in range(5):
    model.predict(images_batch, verbose=0)


n_runs = 100
start = time.time()

for _ in range(n_runs):
    model.predict(images_batch, verbose=0)

end = time.time()

# Zeit pro Bild, nicht nur pro Batch:
avg_time_per_image = (end - start) / (n_runs * len(images_batch))

print(f"Durchschnittliche Laufzeit pro Bild: {avg_time_per_image * 1000:.2f} ms")
y_true, y_pred, y_conf = [], [], []

def get_file_size_mb(path):
    return os.path.getsize(path) / (1024**2)
print("=== Size ===")
size_mb = get_file_size_mb("../../modelle/final/MobileNetV2FT.keras")
print(f"{size_mb:.2f} MB")

print("=== Size ohne Optimierer ===")
parameter, size_mib = modellgroesse_ohne_optimizer(model_finetuned)

print(f"Modellparameter: {parameter:,}")
print(f"{size_mib:.2f} MB")

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))
    y_conf.extend(np.max(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_conf = np.array(y_conf)

print("=== Classification Report ===")
print(classification_report(y_true, y_pred, target_names=class_names,
    digits=4))
cm = confusion_matrix(y_true, y_pred)

print(cm)


👾Pretained finetuned CNN👾
=== Time ===
Durchschnittliche Laufzeit pro Bild: 29.83 ms
=== Size ===
27.23 MB
=== Size ohne Optimierer ===
Modellparameter: 2,591,554
9.89 MB
=== Classification Report ===
                precision    recall  f1-score   support

Apple__Healthy     0.9759    1.0000    0.9878       284
 Apple__Rotten     1.0000    0.9840    0.9919       437

      accuracy                         0.9903       721
     macro avg     0.9880    0.9920    0.9899       721
  weighted avg     0.9905    0.9903    0.9903       721

[[284   0]
 [  7 430]]


In [31]:

import time
import json
import joblib
import numpy as np
import tensorflow as tf
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile
from sklearn.metrics import classification_report, confusion_matrix

RF_PATH = Path("../../modelle/test/mobilenet_v2_randomforest.zip")

with TemporaryDirectory() as temporary:
    folder = Path(temporary)

    with ZipFile(RF_PATH) as bundle:
        # Unkomprimierte Dateigroessen der beiden gespeicherten Modellteile.
        extractor_size_mb = bundle.getinfo("mobilenet_v2_extractor.keras").file_size / (1024 ** 2)
        rf_size_mb = bundle.getinfo("random_forest.joblib").file_size / (1024 ** 2)
        for name in (
            "mobilenet_v2_extractor.keras",
            "random_forest.joblib",
            "metadata.json",
        ):
            (folder / name).write_bytes(bundle.read(name))

    rf_extractor = tf.keras.models.load_model(
        folder / "mobilenet_v2_extractor.keras",
        compile=False,
    )
    model_randomforest = joblib.load(folder / "random_forest.joblib")
    rf_metadata = json.loads(
        (folder / "metadata.json").read_text(encoding="utf-8")
    )

assert list(class_names) == rf_metadata["class_names"], (
    "Die Klassenreihenfolge des Datensatzes passt nicht zum Random Forest."
)

print("\n========================================")
print("👾 Modell: Random Forest + MobileNetV2 👾")

# Ein einzelnes Testbild bzw. ein kleiner Batch für die Zeitmessung
images_batch, labels_batch = next(iter(test_ds))

# Wichtig: Features aus dem MobileNet-Extractor erzeugen
features_batch = rf_extractor.predict(images_batch, verbose=0)

print("\n=== Time ===")

# Warm-up: Feature-Extraktion + Random-Forest-Vorhersage
for _ in range(5):
    features = rf_extractor.predict(images_batch, verbose=0)
    model_randomforest.predict(features)

n_runs = 100
start = time.perf_counter()

for _ in range(n_runs):
    features = rf_extractor.predict(images_batch, verbose=0)
    prediction = model_randomforest.predict(features)

end = time.perf_counter()

# Zeit pro Bild, nicht nur pro Batch:
avg_time_per_image = (end - start) / (n_runs * len(images_batch))

print(f"Durchschnittliche Laufzeit pro Bild: {avg_time_per_image * 1000:.2f} ms")
print("\n=== Size ===")

total_size_mb = extractor_size_mb + rf_size_mb

print(f"MobileNetV2 Feature Extractor: {extractor_size_mb:.2f} MB")
print(f"Random Forest:                {rf_size_mb:.2f} MB")
print(f"Gesamtmodell:                 {total_size_mb:.2f} MB")
print("\n=== Classification Report ===")

y_true = []
y_pred = []
y_conf = []

for images, labels in test_ds:

    # 1. Bilder in Feature-Vektoren umwandeln
    features = rf_extractor.predict(images, verbose=0)

    # 2. Random Forest klassifizieren lassen
    preds = model_randomforest.predict(features)

    # 3. Wahrscheinlichkeiten für die Konfidenz
    probs = model_randomforest.predict_proba(features)

    y_true.extend(labels.numpy())
    y_pred.extend(preds)
    y_conf.extend(np.max(probs, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_conf = np.array(y_conf)

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=4
))
cm = confusion_matrix(y_true, y_pred)

print(cm)



👾 Modell: Random Forest + MobileNetV2 👾

=== Time ===
Durchschnittliche Laufzeit pro Bild: 33.88 ms

=== Size ===
MobileNetV2 Feature Extractor: 9.17 MB
Random Forest:                7.72 MB
Gesamtmodell:                 16.89 MB

=== Classification Report ===
                precision    recall  f1-score   support

Apple__Healthy     0.9467    0.9861    0.9660       288
 Apple__Rotten     0.9905    0.9630    0.9766       433

      accuracy                         0.9723       721
     macro avg     0.9686    0.9746    0.9713       721
  weighted avg     0.9730    0.9723    0.9723       721

[[284   4]
 [ 16 417]]


In [34]:
import time
from pathlib import Path
from tempfile import TemporaryDirectory

# Identischer ImageNet-Extractor wie in CNN_mit_XGBoost.ipynb.
xgb_base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,       # letzten Dense-Layer weglassen
    weights="imagenet",
)
xgb_base_model.trainable = False  # eingefroren — nur Feature Extraktion

xgb_extractor = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Rescaling(scale=1./127.5, offset=-1.0, name="mobilenet_preprocess"),
    xgb_base_model,
    layers.GlobalAveragePooling2D(),   # → (batch, 1280)
], name="MobileNetV2_FeatureExtractor")

XGB_PATH = Path("../../modelle/final/mobilenet_xgboost.json")

def get_file_size_mb(path):
    return os.path.getsize(path) / (1024 ** 2)
print("\n========================================")
print("👾 Modell: XGBoost + MobileNetV2 Features 👾")

images_batch, _ = next(iter(test_ds))
images_batch = images_batch[0:1]

print("\n=== Time (realistisch, mit model()) ===")

# Warm-up — WICHTIG bei model()-Aufruf, da TF beim ersten Call
# den Graphen tracen/kompilieren muss
for _ in range(10):
    features = xgb_extractor(images_batch, training=False).numpy()
    _ = xgb_model.predict(features)

n_runs = 100
mobilenet_times = []
xgb_times = []

for _ in range(n_runs):
    t0 = time.perf_counter()
    features = xgb_extractor(images_batch, training=False).numpy()
    t1 = time.perf_counter()
    prediction = xgb_model.predict(features)
    t2 = time.perf_counter()

    mobilenet_times.append(t1 - t0)
    xgb_times.append(t2 - t1)

avg_mobilenet = np.mean(mobilenet_times) * 1000
avg_xgb = np.mean(xgb_times) * 1000
avg_total = avg_mobilenet + avg_xgb

print(f"MobileNet Feature-Extraktion: {avg_mobilenet:.2f} ms (± {np.std(mobilenet_times)*1000:.2f} ms)")
print(f"XGBoost Prediction:            {avg_xgb:.2f} ms (± {np.std(xgb_times)*1000:.2f} ms)")
print(f"Gesamt pro Bild:                {avg_total:.2f} ms")
print("\n=== Size ===")

# Dateigroesse des tatsaechlich verwendeten Extractors, ohne dauerhafte Datei.
with TemporaryDirectory() as temporary:
    extractor_path = Path(temporary) / "mobilenet_v2_extractor.keras"
    xgb_extractor.save(extractor_path)
    extractor_size_mb = get_file_size_mb(extractor_path)
xgb_size_mb = get_file_size_mb(XGB_PATH)

print(f"MobileNetV2 Feature Extractor: {extractor_size_mb:.2f} MB")
print(f"XGBoost-Modell:               {xgb_size_mb:.2f} MB")
print(f"Gesamtmodell:                  {extractor_size_mb + xgb_size_mb:.2f} MB")


print("\n=== Classification Report ===")

y_true = []
y_pred = []
y_conf = []

for images, labels in test_ds:

    # Bilder -> MobileNet-Features
    features = xgb_extractor.predict(images, verbose=0)

    # XGBoost-Klassifikation
    preds = xgb_model.predict(features)

    # Klassifikationswahrscheinlichkeiten
    probs = xgb_model.predict_proba(features)

    y_true.extend(labels.numpy())
    y_pred.extend(preds.astype(int))
    y_conf.extend(np.max(probs, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_conf = np.array(y_conf)

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=4

))


👾 Modell: XGBoost + MobileNetV2 Features 👾

=== Time (realistisch, mit model()) ===
MobileNet Feature-Extraktion: 269.77 ms (± 41.75 ms)
XGBoost Prediction:            5.19 ms (± 18.32 ms)
Gesamt pro Bild:                274.96 ms

=== Size ===
MobileNetV2 Feature Extractor: 9.17 MB
XGBoost-Modell:               0.42 MB
Gesamtmodell:                  9.59 MB

=== Classification Report ===
                precision    recall  f1-score   support

Apple__Healthy     0.9698    0.9830    0.9764       294
 Apple__Rotten     0.9882    0.9789    0.9835       427

      accuracy                         0.9806       721
     macro avg     0.9790    0.9810    0.9799       721
  weighted avg     0.9807    0.9806    0.9806       721



Die Metriken weichen von immer etwas ab. Für die Vision Transformer wurde in einem anderen Notebook gemessen. Die Modelle waren dabei zu Speicheraufwendig um sie hochzuladen.

In [32]:
model_finetuned.summary()

Model: "MobileNetV2_Agrarkontrolle"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenet_preprocess            │ (None, 224, 224, 3)    │             0 │
│ (Rescaling)                     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           514 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,976,456 (26.61 MB)

 Trainable params: 2,192,450 (8.36 MB)

 Non-trainable params: 399,104 (1.52 MB)

 Optimizer params: 4,384,902 (16.73 MB)

In [33]:
model_best_CNN.summary()

Model: "MobileNetV2_Agrarkontrolle"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenet_preprocess            │ (None, 224, 224, 3)    │             0 │
│ (Rescaling)                     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           514 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,253,576 (12.41 MB)

 Trainable params: 331,010 (1.26 MB)

 Non-trainable params: 2,260,544 (8.62 MB)

 Optimizer params: 662,022 (2.53 MB)

In [ ]:
#speicher = total params - optimiezer params